# XGBoost v2.3 — High Attack Recall + Low False Positive Rate

## What Changed from v2.2 → v2.3

| Parameter | v2.2 | v2.3 | Why |
|---|---|---|---|
| `scale_pos_weight` | `0.8` | **`1.5`** | Weight attacks 1.5x more than benign |
| `BETA` | `2.0` | **`3.0`** | Threshold tuned even harder for attack recall |
| `dns_queries_per_second` in LOG | ✅ Yes | **❌ No** | Benign CSV had MORE queries/sec than attack — log was hiding attack signal |
| `min_child_weight` | `5` | **`3`** | Finer splits for subtle attack sub-patterns |
| `n_estimators` | `800` | **`1000`** | More trees = better generalisation |

## Root Cause of Low Attack Recall in v2.2
The diagnostic showed:
- Benign CSV: `dns_queries_per_second = 30.90` (script was too aggressive)
- Attack CSV: `dns_queries_per_second = 10.51` (script was too mild)

After `log1p()`, benign=3.46 vs attack=2.45 — the model thought LOW dns query rate = ATTACK.
**Fix**: keep `dns_queries_per_second` in raw form so the model can use it as a direct signal.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, fbeta_score
)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
print('[OK] Libraries loaded')

## 1. Load Data

In [ ]:
FILE_PATH = r"C:\Users\shenal\Downloads\reseraach\PCAPS_Used\Final_Balanced_Attack_and_Benign\Final_balanced_Attack_and_Benign_new_Shuffled.csv"

print(f'[INFO] Loading: {FILE_PATH}')
df = pd.read_csv(FILE_PATH)
print(f'[OK] Rows: {len(df):,}  Cols: {len(df.columns)}')
print(df['label'].value_counts())
df.head(3)

## 2. Preprocessing + Selective Log-Transform

In [ ]:
COLS_TO_DROP = ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol_number']
df = df.drop(columns=COLS_TO_DROP, errors='ignore')
df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

LABEL_MAP = {'BENIGN': 0, 'ATTACK': 1}
df['label'] = df['label'].str.upper().map(LABEL_MAP).fillna(0).astype(int)
print(f'Labels: {dict(df["label"].value_counts())}')

PROTOCOL_CLASSES = ['DOH', 'DOT', 'TRADITIONAL', 'UNKNOWN', 'TCP', 'UDP']
le_proto = LabelEncoder()
le_proto.fit(PROTOCOL_CLASSES)
df['protocol'] = df['protocol'].astype(str).apply(
    lambda x: x if x in PROTOCOL_CLASSES else 'UNKNOWN'
)
df['protocol'] = le_proto.transform(df['protocol'])

# ================================================================
# SELECTIVE LOG-TRANSFORM — v2.3 KEY CHANGE
#
# ONLY transform features where BENIGN traffic is naturally high
# due to streaming (YouTube, Netflix, Discord).
#
# DO NOT transform features that are primary attack signals:
#   - dns_queries_per_second: attack diagnostic showed benign CSV
#     had MORE queries/sec than attack CSV. Log-transform was
#     inverting the signal and hurting attack recall.
#   - dns_amplification_factor: large values = attack (keep raw)
#   - query_response_ratio: large values = attack (keep raw)
# ================================================================
LOG_FEATURES = [
    'bwd_packets_per_sec',    # Was 72% importance — MUST be compressed
    'flow_bytes_per_sec',     # Streaming creates high values
    'flow_packets_per_sec',   # Streaming creates high values
    'fwd_packets_per_sec',    # Streaming creates high values
    'total_fwd_packets',      # Large for long streaming sessions
    'total_bwd_packets',      # Large for long streaming sessions
    # REMOVED from v2.2: dns_queries_per_second
    #   Benign CSV had mean=30.90, Attack CSV had mean=10.51
    #   Log-transform was making attacks look like benign!
    # REMOVED from v2.2: dns_amplification_factor
    #   Large value = attack signal — keep raw
]

print('[LOG-TRANSFORM v2.3] Only compressing streaming-related features...')
for col in LOG_FEATURES:
    if col in df.columns:
        df[col] = np.log1p(df[col].clip(lower=0))
        print(f'  log1p({col})')

print('\n[KEPT RAW — Attack signals]:')
for col in ['dns_queries_per_second', 'dns_amplification_factor',
            'query_response_ratio', 'dns_any_query_ratio']:
    if col in df.columns:
        print(f'  {col} (mean={df[col].mean():.2f}, max={df[col].max():.2f})')

print('\n[OK] Preprocessing complete')

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
n_benign = (y_train == 0).sum()
n_attack = (y_train == 1).sum()
print(f'Train: BENIGN={n_benign:,}  ATTACK={n_attack:,}  ratio={n_attack/n_benign:.2f}')

## 4. XGBoost Training — v2.3 High Recall Config

### Tuning Guide
| Symptom | Action |
|---|---|
| Attack recall < 85% | Raise `scale_pos_weight` → try `2.0` |
| BENIGN recall < 85% | Lower `scale_pos_weight` → try `1.2` |
| Both recalls low | Increase `n_estimators` to 1500 |

In [ ]:
# ========================
# PRIMARY TUNING KNOB
# > 1.0 = attack-sensitive
# = 1.0 = balanced
# < 1.0 = benign-safe
# ========================
SCALE_POS_WEIGHT = 1.5

model = xgb.XGBClassifier(
    objective          = 'binary:logistic',
    eval_metric        = ['logloss', 'auc'],
    use_label_encoder  = False,

    # Class balance — v2.3: weight attacks 1.5x
    scale_pos_weight   = SCALE_POS_WEIGHT,

    # Tree architecture
    n_estimators       = 1000,         # +200 over v2.2
    learning_rate      = 0.03,
    max_depth          = 6,            # +1 over v2.2 — allows deeper attack patterns
    min_child_weight   = 3,            # -2 from v2.2 — finer splits
    gamma              = 0.05,

    # Regularisation
    subsample          = 0.8,
    colsample_bytree   = 0.7,
    colsample_bylevel  = 0.7,
    reg_alpha          = 0.05,
    reg_lambda         = 1.0,

    tree_method        = 'hist',
    random_state       = 42,
)

print(f'[TRAIN] scale_pos_weight={SCALE_POS_WEIGHT}, max_depth=6, min_child_weight=3')
model.fit(
    X_train, y_train,
    eval_set = [(X_train, y_train), (X_test, y_test)],
    verbose  = 100,
)
print('[DONE] Training complete.')

## 5. Threshold Tuning (beta=3.0 — Maximum Attack Recall)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

# beta=3.0: attack recall is 9x more important than precision
# This will pick a very low threshold to catch as many attacks as possible
BETA = 3.0

thresholds = np.arange(0.05, 0.95, 0.01)
best_thresh, best_score = 0.5, 0
results = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    score    = fbeta_score(y_test, y_pred_t, beta=BETA, zero_division=0)
    results.append((t, score))
    if score > best_score:
        best_score, best_thresh = score, t

print(f'[TUNING] Optimal threshold = {best_thresh:.2f}  (F-{BETA} = {best_score:.4f})')

scores = [r[1] for r in results]
plt.figure(figsize=(10, 4))
plt.plot(thresholds, scores, color='steelblue')
plt.axvline(best_thresh, color='red',  linestyle='--', label=f'Best={best_thresh:.2f}')
plt.axvline(0.5,         color='gray', linestyle=':',  label='Default=0.50')
plt.xlabel('Threshold')
plt.ylabel(f'F-beta (beta={BETA})')
plt.title('v2.3 Threshold Tuning — Attack Recall Optimised')
plt.legend(); plt.show()

## 6. Balance Report

In [ ]:
y_pred = (y_prob >= best_thresh).astype(int)

print(f'=== CLASSIFICATION REPORT (threshold={best_thresh:.2f}) ===')
print(classification_report(y_test, y_pred, target_names=['BENIGN','ATTACK'], digits=4))

auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['BENIGN','ATTACK'], yticklabels=['BENIGN','ATTACK'])
plt.title(f'Confusion Matrix @ threshold={best_thresh:.2f} (v2.3)')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
benign_recall  = tn / (tn + fp) if (tn + fp) > 0 else 0
attack_recall  = tp / (tp + fn) if (tp + fn) > 0 else 0
false_pos_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f'\n===============================')
print(f' v2.3 BALANCE REPORT')
print(f'===============================')
print(f' BENIGN Recall  : {benign_recall*100:6.1f}%  (target: >90%)')
print(f' ATTACK Recall  : {attack_recall*100:6.1f}%  (target: >90%)')
print(f' False Pos Rate : {false_pos_rate*100:6.1f}%  (target: <10%)')
print(f' ROC-AUC        : {auc:.4f}           (target: >0.97)')
print(f'===============================')

if attack_recall >= 0.90 and benign_recall >= 0.90:
    print('\n[EXCELLENT] Both recalls above 90%! Ready for evaluation.')
elif attack_recall < 0.85:
    print('\n[ADVICE] Attack recall still low.')
    print('  Option 1: Raise scale_pos_weight to 2.0')
    print('  Option 2: Add more real attack captures to training data')
elif benign_recall < 0.85:
    print('\n[ADVICE] Too many false positives.')
    print('  Option 1: Lower scale_pos_weight to 1.2')
    print('  Option 2: Add your 1-hour real YouTube/Netflix capture to training')

## 7. Feature Importance — Check Attack Signal Features Are Now Used

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns)
top20 = importance.nlargest(20)

plt.figure(figsize=(10, 7))
top20.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 20 Feature Importances — XGBoost v2.3')
plt.xlabel('Importance Score'); plt.tight_layout(); plt.show()

print('\n--- Top 10 Features ---')
for feat, imp in top20.head(10).items():
    bar = '#' * int(imp * 100)
    print(f'  {feat:<35} {imp:.4f}  {bar}')

# Check if DNS attack features are now being used
dns_features = ['dns_queries_per_second', 'dns_amplification_factor',
                'query_response_ratio', 'dns_any_query_ratio', 'dns_txt_query_ratio']
print('\n--- DNS Attack Feature Importance ---')
for feat in dns_features:
    imp = importance.get(feat, 0)
    status = '[GOOD]' if imp > 0.02 else '[LOW]'
    print(f'  {status} {feat:<35} {imp:.4f}')

bwd_imp = importance.get('bwd_packets_per_sec', 0)
print(f'\n[CHECK] bwd_packets_per_sec = {bwd_imp:.4f}  (was 72% in v2.1, target: <20%)')

## 8. Save Model Bundle

In [ ]:
bundle = {
    'model':             model,
    'threshold':         best_thresh,
    'log_features':      LOG_FEATURES,
    'protocol_classes':  PROTOCOL_CLASSES,
    'scale_pos_weight':  SCALE_POS_WEIGHT,
    'beta':              BETA,
    'version':           'v2.3',
    'notes':             'dns_queries_per_second kept raw; scale_pos_weight=1.5; beta=3.0',
}

with open('xgb_model_v2.3.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print(f'[SAVED] xgb_model_v2.3.pkl')
print(f'  threshold        = {best_thresh:.2f}')
print(f'  scale_pos_weight = {SCALE_POS_WEIGHT}')
print(f'  beta             = {BETA}')
print(f'  ATTACK Recall    = {attack_recall*100:.1f}%')
print(f'  BENIGN Recall    = {benign_recall*100:.1f}%')
print(f'  ROC-AUC          = {auc:.4f}')
print()
print('To TEST: Update MODEL_PATH in test_xgb_model_v2.1.py to xgb_model_v2.3.pkl')